# Phase 3 — ML pipeline

Presentation notebook for the SPY vol-crush short-straddle classifier.

Primary target: `profitable_5d`. Secondary robustness: `profitable_10d`. `profitable_21d` is exploratory only (its labels are mostly intrinsic settlements and are structurally different — see Phase 2).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path('.').resolve()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import ml_pipeline as mp

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. Run the full pipeline

`mp.run_pipeline()` loads data, splits temporally, fits all four models, calibrates on val, picks thresholds on val, evaluates on all three splits, and writes `metrics.csv`, `predictions.csv`, and pickled models.

In [ ]:
result = mp.run_pipeline(target=mp.PRIMARY_TARGET, pnl_col=mp.PRIMARY_PNL_COL, save=True)

train, val, test = result['train'], result['val'], result['test']
metrics_df      = result['metrics_df']
preds_df        = result['preds_df']
calibrated      = result['calibrated']
lr_thr, lgb_thr = result['thresholds']['logreg'], result['thresholds']['lgbm']
rule_drift      = result['rule_drift']

## 2. Classification metrics per model per split

In [ ]:
cls_cols = ['accuracy','mcc','auc_pr','auc_roc','brier','log_loss','precision','recall','f1','tn','fp','fn','tp']
tbl = metrics_df.pivot_table(index='model', columns='split', values=cls_cols, observed=True, dropna=False)
tbl = tbl.reorder_levels([1, 0], axis=1).sort_index(axis=1, level=0)
with pd.option_context('display.max_columns', None, 'display.width', 250):
    for split in ['train','val','test']:
        print(f'\n--- {split.upper()} ---')
        sub = metrics_df[metrics_df['split'] == split].set_index('model')[cls_cols]
        print(sub.round(4).to_string())

## 3. Trade economics on Test — the decision-relevant table

In [ ]:
trade_cols = [
    'n_taken','coverage_pct','hit_rate_taken',
    'mean_pnl_pct_taken','median_pnl_pct_taken','std_pnl_pct_taken',
    'sum_pnl_pct_taken','worst_pnl_pct_taken','max_drawdown_taken',
    'sharpe_overlapping','sharpe_non_overlapping','n_non_overlapping_trades',
]
print('Trade economics (Test set, primary target = profitable_5d):')
print(metrics_df[metrics_df['split']=='test'].set_index('model')[trade_cols].round(4).to_string())

## 4. Disaster diagnostics on Test

Disaster = any test day with `pnl_pct_5d < -0.25`. For each model:
- how many of those disasters did we AVOID (correctly predict not-profitable)?
- of the ones that SLIPPED THROUGH (model flagged profitable, trade was taken, P&L < -25%), what was the mean and worst P&L?

A high avoidance rate alone is not enough. A model that avoids 90% of disasters but lets through a -80% trade is still dangerous.

In [ ]:
disaster_cols = [
    'n_disasters_total','n_disasters_avoided','pct_disasters_avoided',
    'n_disasters_slipped','mean_pnl_slipped','worst_pnl_slipped',
    'n_skipped','mean_pnl_pct_skipped','hit_rate_skipped',
]
print('Disaster + skipped-set diagnostics (Test set):')
print(metrics_df[metrics_df['split']=='test'].set_index('model')[disaster_cols].round(4).to_string())

## 5. VRP rule: train-threshold → test hit-rate sanity check

Phase 2 found an in-sample hit rate of ~78% when selecting trades in the top-VRP quintile. If the Train-derived threshold applied to Test deviates from that by more than 10 pp, that indicates regime shift or an in-sample artifact.

In [ ]:
vrp = result['models']['vrp_rule']
test_mask = (test['vrp_30d'] >= vrp.threshold)
test_n   = int(test_mask.sum())
test_hit = float(test[test_mask][mp.PRIMARY_TARGET].mean()) if test_n else float('nan')
drift_pp = (test_hit - 0.78) * 100
print(f'Train q80 threshold : {vrp.threshold:.4f}')
print(f'Train-in-sample hit : {vrp.train_pos_rate:.3%}  (on Train rows with vrp_30d >= q80)')
print(f'Test rule hit rate  : {test_hit:.3%}  on {test_n} trades')
print(f'Drift vs Phase-2 78%: {drift_pp:+.1f} pp  {"[FLAG]" if abs(drift_pp) > 10 else "[OK]"}')

## 6. Confusion matrices per model on Test

In [ ]:
from sklearn.metrics import confusion_matrix

models_for_viz = ['always_positive', 'vrp_rule', 'logreg', 'lgbm']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, name in zip(axes, models_for_viz):
    y_true = preds_df['y_true'].to_numpy()
    y_pred = preds_df[f'{name}_pred'].to_numpy()
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    im = ax.imshow(cm, cmap='Blues')
    for (i,j), v in np.ndenumerate(cm):
        ax.text(j, i, f'{v}', ha='center', va='center',
                color='white' if v > cm.max()/2 else 'black', fontsize=14)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Pred 0','Pred 1'])
    ax.set_yticklabels(['True 0','True 1'])
    ax.set_title(f'{name}')
fig.suptitle('Confusion matrices — Test set')
plt.tight_layout(); plt.show()

## 7. Calibration (reliability) plot — LogReg and LGBM on Test

Watch specifically for high-probability predictions falling below the diagonal in the upper right — that would be the Val→Test calibration shift I flagged pre-training.

In [ ]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(7, 6))
for name, color in [('logreg', 'tab:blue'), ('lgbm', 'tab:orange')]:
    y_true  = preds_df['y_true'].to_numpy()
    y_proba = preds_df[f'{name}_proba'].to_numpy()
    prob_true, prob_pred = calibration_curve(y_true, y_proba, n_bins=10, strategy='quantile')
    ax.plot(prob_pred, prob_true, 'o-', color=color, label=f'{name} (n_bins=10)')
ax.plot([0,1],[0,1],'k--', lw=0.8, label='y = x')
ax.set_xlabel('Predicted probability (calibrated)')
ax.set_ylabel('Observed fraction positive')
ax.set_title('Reliability diagram — Test set')
ax.legend(); ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout(); plt.show()

## 8. Predicted-probability distribution split by actual outcome

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, name, thr in zip(axes, ['logreg','lgbm'], [lr_thr, lgb_thr]):
    y_true  = preds_df['y_true'].to_numpy()
    y_proba = preds_df[f'{name}_proba'].to_numpy()
    ax.hist(y_proba[y_true==1], bins=25, alpha=0.55, color='tab:green', label='actual winners')
    ax.hist(y_proba[y_true==0], bins=25, alpha=0.55, color='tab:red',   label='actual losers')
    ax.axvline(thr, color='k', ls='--', lw=1, label=f'threshold = {thr:.2f}')
    ax.set_title(f'{name} — calibrated probabilities on Test')
    ax.set_xlabel('P(profitable)')
    ax.legend()
plt.tight_layout(); plt.show()

## 9. LGBM SHAP summary and feature importance

SHAP values computed on **test** predictions with the fitted LGBM — no training on test data.

In [ ]:
import shap

lgbm_raw = result['models']['lgbm_raw']
X_test = test[mp.FEATURE_COLS]
explainer = shap.TreeExplainer(lgbm_raw)
sv = explainer.shap_values(X_test)
# LightGBM binary: older shap returns list, newer returns 3D/2D array — normalize
if isinstance(sv, list):
    sv_pos = sv[1]
elif isinstance(sv, np.ndarray) and sv.ndim == 3:
    sv_pos = sv[:, :, 1]
else:
    sv_pos = sv

fig = plt.figure(figsize=(9, 7))
shap.summary_plot(sv_pos, X_test, show=False, max_display=15)
plt.title('LGBM SHAP summary — Test set')
plt.tight_layout(); plt.show()

# Top-10 mean |SHAP|
mean_abs = pd.Series(np.abs(sv_pos).mean(axis=0), index=mp.FEATURE_COLS).sort_values(ascending=False)
print('\nTop 15 features by mean |SHAP|:')
print(mean_abs.head(15).round(4).to_string())

## 10. Feature importance bar chart (LGBM gain)

In [ ]:
importances = pd.Series(lgbm_raw.booster_.feature_importance(importance_type='gain'),
                        index=mp.FEATURE_COLS).sort_values(ascending=True).tail(20)
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(importances.index, importances.values, color='tab:blue')
ax.set_xlabel('Gain'); ax.set_title('LGBM feature importance (gain) — top 20')
plt.tight_layout(); plt.show()

## 11. Predicted-probability time series vs. actual P&L on Test

Including an annotation of the April 2025 tariff episode. The question is whether the model *pulled back* (lowered predicted probabilities) as IV spiked — or whether it kept happily predicting profitable through the spike.

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))
dates = preds_df.index
ax1.plot(dates, preds_df['lgbm_proba'],   label='LGBM P(profit)', color='tab:orange', lw=1.1)
ax1.plot(dates, preds_df['logreg_proba'], label='LogReg P(profit)', color='tab:blue',  lw=1.1, alpha=0.7)
ax1.axhline(lgb_thr, color='tab:orange', ls='--', lw=0.8, label=f'LGBM thr = {lgb_thr:.2f}')
ax1.axhline(lr_thr,  color='tab:blue',   ls='--', lw=0.8, label=f'LogReg thr = {lr_thr:.2f}')
ax1.set_ylabel('Predicted P(profitable)'); ax1.set_ylim(0, 1)
ax1.set_title('Predicted probabilities and realised P&L on Test')
ax1.legend(loc='upper left')
ax2 = ax1.twinx()
ax2.bar(dates, preds_df[mp.PRIMARY_PNL_COL], width=1.0, color='gray', alpha=0.3, label='pnl_pct_5d')
ax2.set_ylabel('pnl_pct_5d'); ax2.axhline(0, color='k', lw=0.5)
for d_label, d in [('Liberation Day peak','2025-04-08'), ('Yen unwind','2024-08-05')]:
    ax1.axvline(pd.Timestamp(d), color='red', alpha=0.3, ls='-.')
    ax1.text(pd.Timestamp(d), 0.02, d_label, rotation=90, va='bottom', ha='right', fontsize=8, color='red')
plt.tight_layout(); plt.show()

## 12. Equity curves on Test — model vs. always-positive baseline

In [ ]:
pnl = preds_df[mp.PRIMARY_PNL_COL].fillna(0).to_numpy()
fig, ax = plt.subplots(figsize=(14, 5))
for name, color in [('always_positive','tab:gray'),
                    ('vrp_rule','tab:purple'),
                    ('logreg','tab:blue'),
                    ('lgbm','tab:orange')]:
    take = preds_df[f'{name}_pred'].to_numpy().astype(bool)
    eq = np.cumsum(np.where(take, pnl, 0.0))
    ax.plot(preds_df.index, eq, label=f'{name} (end={eq[-1]:+.2f})', color=color, lw=1.2)
ax.axhline(0, color='k', lw=0.5)
ax.set_ylabel('Cumulative pnl_pct_5d (additive)')
ax.set_title('Equity curves on Test — additive sum of pnl_pct_5d across taken trades')
for d_label, d in [('Liberation Day','2025-04-08'), ('Yen unwind','2024-08-05')]:
    ax.axvline(pd.Timestamp(d), color='red', alpha=0.3, ls='-.')
ax.legend()
plt.tight_layout(); plt.show()

# Report terminal + drawdown per model explicitly
print('\nTerminal cumulative P&L and max drawdown per model (Test):')
for name in ['always_positive','vrp_rule','logreg','lgbm']:
    take = preds_df[f'{name}_pred'].to_numpy().astype(bool)
    eq = np.cumsum(np.where(take, pnl, 0.0))
    peak = np.maximum.accumulate(eq)
    dd = eq - peak
    i_dd = int(np.argmin(dd))
    print(f'  {name:<18s} terminal={eq[-1]:+.3f}  max_dd={dd.min():+.3f} on {preds_df.index[i_dd].date()}')

## 13. April 2025 tariff episode — micro-analysis

For each trading day from 2025-04-03 through 2025-04-15, show predicted probabilities and take/skip decisions per model, plus the actual 5d P&L.

In [ ]:
episode = preds_df.loc['2025-04-03':'2025-04-15', [
    'always_positive_pred','vrp_rule_pred','vrp_rule_proba',
    'logreg_proba','logreg_pred','lgbm_proba','lgbm_pred','y_true',
    mp.PRIMARY_PNL_COL,
]].copy()
print('April 2025 tariff episode decisions:')
with pd.option_context('display.max_columns', None, 'display.width', 200):
    print(episode.round(4).to_string())

# Per-model coverage and P&L over the episode
print('\nEpisode (2025-04-03..15) summary per model:')
for name in ['always_positive','vrp_rule','logreg','lgbm']:
    take = episode[f'{name}_pred'].to_numpy().astype(bool)
    pnl_e = episode[mp.PRIMARY_PNL_COL].fillna(0).to_numpy()
    taken_pnl = pnl_e[take]
    print(f'  {name:<18s} took {take.sum():>2d}/{len(take)} trades, '
          f'sum_pnl={taken_pnl.sum():+.3f}, '
          f'mean_pnl={taken_pnl.mean() if take.any() else float("nan"):+.3f}')

## 14. Exploratory — profitable_21d

Separate run on the 21d target. Labels are structurally different (most are intrinsic settlements, per Phase 2) so the model selection was NOT tuned against this — report it here for completeness only.

In [ ]:
try:
    result_21 = mp.run_pipeline(target='profitable_21d', pnl_col='pnl_pct_21d',
                                 horizon_days=21, save=False)
    m21 = result_21['metrics_df']
    print('\n21d target — classification metrics (Test):')
    print(m21[m21['split']=='test'].set_index('model')[
        ['accuracy','mcc','auc_pr','precision','recall','coverage_pct','hit_rate_taken',
         'mean_pnl_pct_taken','sum_pnl_pct_taken','pct_disasters_avoided',
         'mean_pnl_slipped']].round(4).to_string())
except Exception as e:
    print(f'21d exploratory run skipped: {e}')